# Chat with a twin: both halves, read and written

[Tutorial 07](../07-integrate-graph-and-timeseries/integrate-graph-and-timeseries.ipynb) joined a
graph to the databases it points at and drove the join by hand — a SPARQL locator, a SQL query, and
arithmetic in Python. Every step was deterministic and every step was yours to write.

This notebook puts a model in front of exactly that, and in exactly three places: **writing the
locator**, **writing the SQL**, and **naming the KPIs in a plan**. Nothing else moves.
`Tool.TwinTargets` still checks the files, `SQL.Validate` still compiles the query against every
one of them, and `Tool.TwinKPIObjects` still does all the arithmetic. A number that lands in the
graph is still a number no model touched.

What that buys is a single entry point. *"How many spaces are in the library?"* and *"How much
electricity did it use?"* are the same kind of question to whoever is asking and completely
different pipelines underneath, and here they are both just something you typed.

It is the twin-side counterpart of [tutorial
04](../04-chat-with-graph/chat-with-graph.ipynb) and [tutorial
06](../06-chat-with-timeseries/chat-with-timeseries.ipynb), and it does one thing neither of them
can: **an edit that reads one half and writes the other.** Ask it to work out each building's
electricity use per square metre and record it, and the figure becomes part of the graph.

The route:

1. a twin to talk to, and the two grounding blocks
2. which half does a question need? — `Tool.TwinRoute`
3. one question, end to end — six steps, four model calls
4. a question about the estate never opens a database
5. a conversation, where "and which of the two?" means something
6. the derive edit — the readings read, the graph written
7. the terminal chat

**Prerequisites**

```bash
pip install "btwin[llm,rdf]"
export OPENROUTER_API_KEY=sk-or-...
```

**Two things to expect**

- **Every cell that calls a model is billed to your key.** The `CostMeter` total near the end says
  exactly what the committed run cost.
- **The outputs will not reproduce exactly.** The model is free to word an answer, and to write a
  query, differently on your run. Where it matters, this notebook checks its work rather than
  trusting the sentence.

In [1]:
import random
from pathlib import Path

import pandas as pd

import btwin
from btwin import (RDF, CostMeter, Cycle, Document, LLM, Observation, Property,
                   PropertySet, Serialization, SpatialElement, Tool)

OUTPUT = Path("output")
OUTPUT.mkdir(exist_ok=True)
DB_DIR = OUTPUT / "databases"
DB_DIR.mkdir(exist_ok=True)

BASE_IRI = "https://example.org/harbourside/"
BASE_PATH = Path(".")          # what the graph's relative FilePath values resolve against
TABLE = "observations"
YEAR = 2025

llm = LLM.Constructor()
meter = CostMeter()

print("BTwin", btwin.__version__)
print("model:", llm.model_name)

C:\Users\massa\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Couldn't import dot_parser, loading of dot files will not be possible.


BTwin 0.5.7
model: google/gemini-2.5-flash-lite


## 1. A twin to talk to

Tutorial 07's estate, rebuilt in one cell so this notebook stands on its own: three buildings with
their spaces and floor areas, six SQLite files of monthly meter readings, and a `btwin:Document`
per database carrying the path, the table and the sensor.

The *why* of every line below is
[tutorial 07](../07-integrate-graph-and-timeseries/integrate-graph-and-timeseries.ipynb); the seed
is the same, so the figures are the same too. **No model is involved in building it.**

In [2]:
ESTATE = {
    "B1": ("Library", {
        "F0": ("Ground Floor", [("Reading Room", "study"), ("Stacks", "storage")]),
        "F1": ("First Floor",  [("Study Carrels", "study"), ("Quiet Room", "study")]),
    }),
    "B2": ("Science Block", {
        "F0": ("Ground Floor", [("Prep Room", "support"), ("Store", "storage")]),
        "F1": ("First Floor",  [("Wet Lab A", "laboratory"), ("Wet Lab B", "laboratory")]),
    }),
    "B3": ("Sports Hall", {
        "F0": ("Ground Floor", [("Main Hall", "hall"), ("Changing Rooms", "support")]),
    }),
}
USES = {"study": (260.0, 3.6, 90), "storage": (180.0, 3.0, 6), "support": (45.0, 2.7, 4),
        "laboratory": (140.0, 3.4, 18), "hall": (860.0, 8.0, 300)}
UTILITIES = {
    "electricity": {"meterCode": "EM", "observedProperty": "ElectricityConsumption",
                    "unit": "kWh", "baseline": (11_000.0, 30_000.0),
                    "season": [1.02, 0.97, 0.94, 0.88, 0.96, 1.14,
                               1.31, 0.83, 1.03, 1.05, 1.01, 1.06]},
    "water":       {"meterCode": "WM", "observedProperty": "WaterConsumption",
                    "unit": "m3", "baseline": (150.0, 800.0),
                    "season": [1.04, 1.06, 1.10, 1.00, 1.05, 0.92,
                               0.80, 0.31, 1.02, 1.14, 1.18, 1.16]},
}


def PSet(uid, name, properties):
    """One IFC property set from (name, value, quantity, unit) tuples."""
    pset = PropertySet.Constructor(uid, name)
    for propertyName, value, quantity, unit in properties:
        PropertySet.SetProperty(pset, Property.Constructor(
            propertyName, value, propertyQuantity=quantity, propertyUnit=unit))
    return pset


# --- the readings -----------------------------------------------------------------------
rng = random.Random(20260902)
databases = {}
for code in ESTATE:
    for utility, spec in UTILITIES.items():
        sensor = f"HB-{code}-{spec['meterCode']}"
        low, high = spec["baseline"]
        baseline = rng.uniform(low, high) / 12.0
        frame = pd.DataFrame(
            [[sensor, spec["observedProperty"], spec["unit"],
              round(baseline * spec["season"][m] * rng.uniform(0.93, 1.07), 2),
              f"{YEAR}-{m + 1:02d}-01T00:00:00Z"] for m in range(12)],
            columns=list(Observation.Template().columns))
        path = DB_DIR / f"HB-{code}-{utility}.db"
        Observation.SQLiteByDF(frame, str(path), TABLE, ifExists="replace")
        databases[(code, utility)] = {
            "path": path, "sensor": sensor, "rows": len(frame), "unit": spec["unit"],
            "observedProperty": spec["observedProperty"]}

# --- the estate, and the documents that point at those files -----------------------------
objects = []
site = SpatialElement.Constructor("site", "bot:Site", "Harbourside Campus")
objects.append(site)

for code, (buildingName, storeys) in ESTATE.items():
    building = SpatialElement.Constructor(code, "bot:Building", buildingName)
    SpatialElement.SetLocationRelationship(building, linkedObject=site)
    objects.append(building)

    for storeyCode, (storeyName, spaces) in storeys.items():
        storeyUID = f"{code}-{storeyCode}"
        storey = SpatialElement.Constructor(
            storeyUID, "bot:Storey", f"{buildingName} - {storeyName}")
        SpatialElement.SetLocationRelationship(storey, linkedObject=building)
        objects.append(storey)
        for number, (spaceName, use) in enumerate(spaces, start=1):
            area, height, people = USES[use]
            spaceUID = f"{storeyUID}-S{number:02d}"
            space = SpatialElement.Constructor(spaceUID, "bot:Space", spaceName)
            SpatialElement.SetLocationRelationship(space, linkedObject=storey)
            pset = PSet(f"{spaceUID}-PSET", "Space Quantities", (
                ("NetFloorArea",  area,   "IfcAreaMeasure",   "m2"),
                ("ClearHeight",   height, "IfcLengthMeasure", "m"),
                ("OccupantCount", people, "IfcCountMeasure",  "people")))
            SpatialElement.SetPSetRelationship(space, pset=pset)
            objects += [space, pset]

    for utility in UTILITIES:
        record = databases[(code, utility)]
        documentUID = f"{code}-{utility}-db"
        document = Document.Constructor(
            documentUID, f"{buildingName} - {utility} readings {YEAR}")
        pset = PSet(f"{documentUID}-PSET", "Database Location", (
            ("FilePath",         str(record["path"]).replace("\\", "/"), "IfcText",   None),
            ("TableName",        TABLE,                      "IfcLabel",        None),
            ("SensorID",         record["sensor"],           "IfcLabel",        None),
            ("ObservedProperty", record["observedProperty"], "IfcLabel",        None),
            ("Unit",             record["unit"],             "IfcLabel",        None),
            ("RowCount",         record["rows"],             "IfcCountMeasure", "rows"),
            ("PeriodStart",      f"{YEAR}-01-01T00:00:00Z",  "IfcDateTime",     None),
            ("PeriodEnd",        f"{YEAR}-12-01T00:00:00Z",  "IfcDateTime",     None)))
        Document.SetPSet(document, pset=pset)
        SpatialElement.SetRelationship(
            building, "btwin:hasDocument", linkedObject=document, validate=False)
        objects += [document, pset]

jsonld = Serialization.JSONLDByObjects(objects)
RDF.ByJSONLD(jsonld=jsonld, savePath=str(OUTPUT / "harbourside.ttl"), baseIRI=BASE_IRI)
graph = RDF.ByTTL(str(OUTPUT / "harbourside.ttl"), baseIRI=BASE_IRI)

print(f"{len(graph)} triples, {len(databases)} databases")

470 triples, 6 databases


Two grounding blocks, because a twin has two halves and each is described by the thing that can
describe it.

The **graph** half is `RDF.SchemaSummary` plus `Tool.TwinPropertyBlock` — classes and predicates,
plus the literals the properties actually hold, which is what a locator's `FILTER` is written
against. The **readings** half is `Observation.SQLiteSchemaSummary` over one representative
database, plus `notes`: the part a flat table cannot say about itself.

`notes` matters more here than it did in tutorial 06, and for a reason peculiar to a twin. The
graph carries the building names; a database carries a sensor id and nothing else. Without the note
below, `HB-B2-EM` is an opaque string.

In [3]:
NOTES = (
    "Every database here holds ONE building's readings for ONE utility: twelve monthly rows "
    "for 2025. Which building a row belongs to is attached afterwards, from the graph.\n"
    "A sensor identifier reads CAMPUS-BUILDING-METER, e.g. 'HB-B2-EM'. The meter codes are "
    "EM = electricity and WM = water."
)

schema = RDF.SchemaSummary(graph)
print(f"{len(schema['terms'])} terms in the graph schema\n")
print(Tool.TwinPropertyBlock(graph))

15 terms in the graph schema



PROPERTY VALUES (what the properties in this graph actually hold -
                 filter on these literals, spelled exactly)
  ClearHeight: 5 numeric value(s), 2.7 to 8
  FilePath: 'output/databases/HB-B1-electricity.db', 'output/databases/HB-B1-water.db', 'output/databases/HB-B2-electricity.db', 'output/databases/HB-B2-water.db', 'output/databases/HB-B3-electricity.db', 'output/databases/HB-B3-water.db'
  NetFloorArea: 5 numeric value(s), 45 to 860
  ObservedProperty: 'ElectricityConsumption', 'WaterConsumption'
  OccupantCount: 5 numeric value(s), 4 to 300
  PeriodEnd: '2025-12-01T00:00:00Z'
  PeriodStart: '2025-01-01T00:00:00Z'
  RowCount: 1 numeric value(s), 12 to 12
  SensorID: 'HB-B1-EM', 'HB-B1-WM', 'HB-B2-EM', 'HB-B2-WM', 'HB-B3-EM', 'HB-B3-WM'
  TableName: 'observations'
  Unit: 'kWh', 'm3'


## 2. Which half does the question need?

`Tool.TwinRoute` is the one step that reads a question before anything has been retrieved, and it
is what lets a single entry point serve two pipelines. It answers `graph`, `readings` or `both`.

It costs one call and it is worth watching on its own, because its three answers are three
genuinely different amounts of work underneath.

In [4]:
for question in [
    "How many spaces does the Science Block have?",
    "How much electricity did the Library use in 2025?",
    "Which building used the most electricity per square metre in 2025?",
    "What is the sports hall's floor area?",
]:
    intent = Tool.TwinRoute(llm, schema["text"], question, meter)
    print(f"  {intent:<10}{question}")

  graph     How many spaces does the Science Block have?


  readings  How much electricity did the Library use in 2025?


  both      Which building used the most electricity per square metre in 2025?


  graph     What is the sports hall's floor area?


The third is the interesting one. *Per square metre* cannot be answered by either half: the
databases have never heard of a floor area, and the graph has never heard of a kilowatt hour. `both`
means the readings are retrieved **and** `Tool.TwinEstateBlock` is put beside them.

`both` is also the safe default when the router's reply cannot be parsed — it gathers everything
either narrower intent would have, so a router that fails is expensive rather than wrong.

## 3. One question, end to end

`Cycle.TwinQueryByPrompt` is six steps, four of which call a model:

1. **`Tool.TwinRoute`** — graph, readings, or both *(model)*
2. **`Tool.TwinWriteLocator`** — the SPARQL that finds *which databases* the question needs
   *(model)*; `SPARQL.Validate` checks it and `Tool.TwinRepairLocator` rewrites it when it fails
3. **`Tool.TwinTargets`** — those rows resolved to files that are actually on disk *(no model)*
4. **`Tool.SQLiteWriteSQL`** — one query for the whole selected set *(model)*
5. **`SQL.Validate`** against every selected database, then each is read and its rows tagged with
   the building they came from *(no model)*
6. **`Tool.TwinAnswer`** — the sentence, written from those rows and the estate block beside them
   *(model)*

The cost is flat in the number of databases: four calls whether the question touches one building
or twenty. `verbose=True` prints each step as it happens.

In [5]:
result = Cycle.TwinQueryByPrompt(
    graph,
    "Which building used the most electricity per square metre of floor area in 2025?",
    basePath=BASE_PATH, llm=llm, schema=schema, meter=meter, notes=NOTES, verbose=True,
)

print(f"\nbot> {result['answer']}")


[router]  2819+6 tokens, $0.000284
[router]  both



[agent 1] 3463+344 tokens, $0.000484
[agent 1] proposed locator:
PREFIX bot: <https://w3id.org/bot#>
PREFIX brick: <https://brickschema.org/schema/Brick#>
PREFIX btwin: <https://example.org/harbourside/btwin#>
PREFIX ifc: <https://standards.buildingsmart.org/IFC/DEV/IFC4/ADD2_TC1/OWL#>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?filePath ?owner ?ownerLabel ?sensor ?property ?unit
WHERE {
  ?owner a bot:Building ;
         rdfs:label ?ownerLabel ;
         btwin:hasDocument ?doc .
  ?doc ifc:HasPropertySets ?pset .
  ?pset ifc:HasProperties ?a . ?a rdfs:label "FilePath" ; ifc:NominalValue ?filePath .
  ?pset ifc:HasProperties ?b . ?b rdfs:label "SensorID" ; ifc:NominalValue ?sensor .
  ?pset ifc:HasProperties ?c . ?c rdfs:label "ObservedProperty" ; ifc:NominalValue ?property .
  ?pset ifc:HasProperties ?d . ?d rdfs:label "Unit" ; ifc:NominalValue ?unit .
  FILTER (?property = "ElectricityConsumption")
}
LIMIT 1


[agent 3] 1459+26 tokens, $0.000156
[agent 3] proposed query:
SELECT SUM(value) AS total_electricity FROM observations WHERE strftime('%Y', timestamp) = '2025'
[agent 5] 3 row(s) from 3 database(s), no model call


[answer]  704+118 tokens, $0.000118

bot> - Library: 28.64 kWh/m2 (27496.22 kWh / 960.0 m2)
- Science Block: 44.86 kWh/m2 (22656.58 kWh / 505.0 m2)
- Sports Hall: 15.64 kWh/m2 (14151.35 kWh / 905.0 m2)

The Science Block used the most electricity per square metre of floor area.


Read it back apart. The two queries are the model's work; everything between them is not.

In [6]:
print("intent:", result["intent"], "\n")
print("--- the locator it wrote (SPARQL, over the graph) ---")
print(result["sparql"].strip())
print("\n--- the databases that resolved to ---")
for target in result["targets"]:
    print(f"  {target['ownerLabel']:<16}{Path(target['path']).name}")
print("\n--- the one query it wrote (SQL, run against each of them) ---")
print(result["sql"].strip())
print("\n--- the rows it was shown ---")
for row in result["rows"]:
    print(" ", "; ".join(f"{k}={v}" for k, v in row.items() if v is not None))
print("\n--- and the estate figures put beside them ---")
print(result["estate"])
print("cost:", CostMeter.Describe(result["usage"]))

intent: both 

--- the locator it wrote (SPARQL, over the graph) ---
PREFIX bot: <https://w3id.org/bot#>
PREFIX brick: <https://brickschema.org/schema/Brick#>
PREFIX btwin: <https://example.org/harbourside/btwin#>
PREFIX ifc: <https://standards.buildingsmart.org/IFC/DEV/IFC4/ADD2_TC1/OWL#>
PREFIX rdf: <http://www.w3.org/1999/02/22-rdf-syntax-ns#>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>

SELECT ?filePath ?owner ?ownerLabel ?sensor ?property ?unit
WHERE {
  ?owner a bot:Building ;
         rdfs:label ?ownerLabel ;
         btwin:hasDocument ?doc .
  ?doc ifc:HasPropertySets ?pset .
  ?pset ifc:HasProperties ?a . ?a rdfs:label "FilePath" ; ifc:NominalValue ?filePath .
  ?pset ifc:HasProperties ?b . ?b rdfs:label "SensorID" ; ifc:NominalValue ?sensor .
  ?pset ifc:HasProperties ?c . ?c rdfs:label "ObservedProperty" ; ifc:NominalValue ?property .
  ?pset ifc:HasProperties ?d . ?d rdfs:label "Unit" ; ifc:NominalValue ?unit .
  FILTER (?property = "ElectricityConsumption")
}
LIMI

This is the whole argument for the design in one screen. The model wrote two queries and one
sentence. It did not write `27,496.2`, it did not write `960.0`, and it did not perform the
division — the first came out of a SQLite file, the second out of the graph, and the answer was
worded from both after the fact.

Which also means the sentence can be checked. Every number in it appears in the rows or the estate
block above, and if one does not, it was invented.

## 4. A question about the estate never opens a database

Routing `graph` is not a shortcut, it is a different pipeline: `Cycle.TwinQueryByPrompt` hands the
question straight to `Cycle.RDFQueryByPrompt` — [tutorial 03's](../03-llm-in-action/llm-in-action.ipynb)
cycle — and none of the database machinery runs at all.

In [7]:
estateOnly = Cycle.TwinQueryByPrompt(
    graph, "How many spaces are there in the Science Block, and how big are they in total?",
    basePath=BASE_PATH, llm=llm, schema=schema, meter=meter, notes=NOTES,
)

print(f"bot> {estateOnly['answer']}\n")
print("intent      :", estateOnly["intent"])
print("databases   :", len(estateOnly["targets"]), "opened")
print("sql         :", repr(estateOnly["sql"]))
print("grounded in :", estateOnly["source"])
print("cost        :", CostMeter.Describe(estateOnly["usage"]))

bot> There are 4 spaces in the Science Block, and their total area is 505.0.

intent      : graph
databases   : 0 opened
sql         : ''
grounded in : []
cost        : 7349+281 tokens, $0.000664


`sql` is empty and `targets` is empty because no file was ever opened: the question went straight
to `Cycle.RDFQueryByPrompt`, the graph cycle of
[tutorial 04](../04-chat-with-graph/chat-with-graph.ipynb) — with the property block put beside the
schema, so that *how big are they* is written against `NetFloorArea` and not against an invented
`Area`. Grounding is not only for the locator.

`source` is the attribution that comes with the graph half: the node IRIs an answer rests on, which
a table cannot give you. It is empty here for a reason worth knowing rather than a fault — the model
asked for a count and a sum, and an aggregate has no node to point at. Attribution needs the query
to project the nodes themselves.

## 5. A conversation

Both cycles above take one self-contained prompt and remember nothing. `Cycle.TwinChatTurn` puts a
conversation on top without changing that, and it routes **twice** rather than once:

- `Tool.ChatRoute` resolves the ellipsis against the transcript and separates small talk from a
  question from a change. This is the only place memory is used, and the restatement it produces is
  what reaches the pipeline — so underneath, everything stays stateless.
- then `Tool.TwinRoute` decides which half that restated question needs.

The turn is stateless too: `history` is not modified, and the returned dict carries a new list with
this turn appended.

In [8]:
history, schemaNow, chains = [], schema, None

for message in [
    "Which buildings are on the Harbourside campus?",
    "How much electricity did they use in 2025?",
    "And which of the two biggest uses more per square metre?",
]:
    turn = Cycle.TwinChatTurn(
        graph, message, basePath=BASE_PATH, baseIRI=BASE_IRI, history=history,
        llm=llm, schema=schemaNow, chains=chains, meter=meter, notes=NOTES,
    )
    history, schemaNow, chains = turn["history"], turn["schema"], turn["chains"]

    print(f"you> {message}")
    print(f"     understood as ({turn['intent']}): {turn['request']}")
    print(f"bot> {turn['answer']}")
    if turn["targets"]:
        print(f"     {len(turn['targets'])} database(s): "
              + ", ".join(t["ownerLabel"] for t in turn["targets"]))
    print(f"     {CostMeter.Describe(turn['usage'])}\n")

you> Which buildings are on the Harbourside campus?
     understood as (question): Which buildings are on the Harbourside campus?
bot> The buildings on the Harbourside campus are the Sports Hall, the Library, and the Science Block.
     7864+265 tokens, $0.000892



you> How much electricity did they use in 2025?
     understood as (question): How much electricity did the Sports Hall, the Library, and the Science Block use in 2025?
bot> The Library used 27496.22 kWh of electricity in 2025. The Science Block used 22656.58 kWh of electricity in 2025. The Sports Hall used 14151.35 kWh of electricity in 2025.
     3 database(s): Library, Science Block, Sports Hall
     8821+551 tokens, $0.001102



you> And which of the two biggest uses more per square metre?
     understood as (question): Which of the Library and the Science Block used more electricity per square metre in 2025?
bot> The Library used 28.64 kWh per square metre, while the Science Block used 44.86 kWh per square metre.
     2 database(s): Library, Science Block
     9010+530 tokens, $0.001113



The second message is the point. *"How much electricity did **they** use?"* is not a question anyone
can answer alone; the router rewrites it against the transcript into one that is, and the query
cycle never sees the pronoun.

The third goes further. *"which of the two biggest"* refers to a set that was never named in any
single message — it exists only across two turns — and *per square metre* then forces the `both`
route. One sentence of ellipsis, and underneath it a locator, three databases, one SQL query and a
sum over the graph.

## 6. The edit that makes a twin worth having as one object

`Tool.TwinEditRoute` sorts a change by **which half it lands in**:

| | |
|---|---|
| `graph` | `Cycle.RDFEditByPrompt` writes a validated SPARQL update — [tutorial 04](../04-chat-with-graph/chat-with-graph.ipynb) |
| `readings` | the databases are located through the graph, then `Cycle.SQLiteEditByPrompt` rehearses a statement against each — [tutorial 06](../06-chat-with-timeseries/chat-with-timeseries.ipynb) |
| `derive` | the databases are **read** and the graph is **written** |

`derive` is the one neither of the other notebooks can do. The readings are retrieved,
`Tool.TwinKPIPlan` says what to record and from which column, `Tool.TwinKPIObjects` does the
arithmetic in Python, and the resulting KPI nodes are added to the graph — [tutorial 07's](../07-integrate-graph-and-timeseries/integrate-graph-and-timeseries.ipynb)
section 9, with the plan written by a model instead of by hand.

**Nothing is written without `confirm`.** It is called with the rehearsed change and decides whether
it lands; `None` never writes. For a derived edit the proposal is a table of building, KPI, value
and unit — with the division that produced each figure beside it, so a number that looks wrong can
be traced before you agree to it rather than after.

In [9]:
def confirm(proposal):
    """Show a rehearsed change and say yes. This is the only thing that lets a write happen."""
    if proposal.get("kind") == "derive":
        print(f"  it would record '{proposal['setName']}':\n")
        for entry in proposal["proposed"]:
            source = (f"{entry['from']} / {entry['dividedBy']}" if entry["dividedBy"]
                      else entry["from"])
            was = f"   was {entry['replaces']}" if entry.get("replaces") is not None else ""
            print(f"    {entry['building']:<16}{entry['kpi']:<28}"
                  f"{entry['value']:>12,.2f} {entry['unit'] or '':<8}({source}){was}")
    else:                                    # a graph edit is a triple diff, a readings
        for triple in proposal.get("removed", []):    # edit a row diff
            print("    -", triple)
        for triple in proposal.get("added", []):
            print("    +", triple)
    print("\n  apply? [y/N] y\n")
    return True


edit = Cycle.TwinChatTurn(
    graph,
    "Work out every building's total 2025 electricity use and its use per square metre, "
    "and record both as KPIs on the buildings.",
    basePath=BASE_PATH, baseIRI=BASE_IRI, history=history, llm=llm, schema=schemaNow,
    chains=chains, meter=meter, confirm=confirm, notes=NOTES,
)
history, schemaNow, chains = edit["history"], edit["schema"], edit["chains"]

print(f"intent: {edit['intent']}   applied: {edit['applied']}   "
      f"{len(edit['added'])} triple(s) added")
print(f"bot> {edit['answer']}")
print(f"     {CostMeter.Describe(edit['usage'])}")

  it would record '2025 Electricity Use':

    Library         AnnualElectricityUse           27,496.22 kWh     (total_electricity_use)
    Library         ElectricityUseIntensity            28.64 kWh/m2  (total_electricity_use / NetFloorArea=960)
    Science Block   AnnualElectricityUse           22,656.58 kWh     (total_electricity_use)
    Science Block   ElectricityUseIntensity            44.86 kWh/m2  (total_electricity_use / NetFloorArea=505)
    Sports Hall     AnnualElectricityUse           14,151.35 kWh     (total_electricity_use)
    Sports Hall     ElectricityUseIntensity            15.64 kWh/m2  (total_electricity_use / NetFloorArea=905)

  apply? [y/N] y

intent: edit:derive   applied: True   6 triple(s) added
bot> Done: 6 KPI(s) recorded on 3 building(s) as '2025 Electricity Use'. The graph now holds 521 triples.
     10512+825 tokens, $0.001106


The graph is now bigger than it was, and the new triples are ordinary graph content — which is what
makes the next turn possible. `Cycle.TwinChatTurn` rebuilds the schema after any confirmed graph
edit, including a derived one, because a derive adds classes the schema has never seen. Ask about
those KPIs and the question routes to `graph`: they are recorded facts about the estate now, not
readings.

In [10]:
after = Cycle.TwinChatTurn(
    graph, "Now which building has the highest electricity use intensity on record?",
    basePath=BASE_PATH, baseIRI=BASE_IRI, history=history, llm=llm, schema=schemaNow,
    chains=chains, meter=meter, notes=NOTES,
)
history, schemaNow, chains = after["history"], after["schema"], after["chains"]

print(f"bot> {after['answer']}")
print(f"     intent={after['intent']}, {len(after['targets'])} database(s) opened\n")

recorded = RDF.Query(graph, """
    PREFIX btwin: <https://example.org/harbourside/btwin#>
    PREFIX eko:   <http://energy.linkeddata.es/em-kpi/ontology#>
    PREFIX ifc:   <https://standards.buildingsmart.org/IFC/DEV/IFC4/ADD2_TC1/OWL#>
    PREFIX rdfs:  <http://www.w3.org/2000/01/rdf-schema#>
    SELECT ?building ?kpi ?value ?unit WHERE {
      ?set a btwin:KPISet ; btwin:hasKPIs ?k ; eko:hasAssociatedObject ?b .
      ?b rdfs:label ?building .
      ?k rdfs:label ?kpi ; ifc:NominalValue ?value .
      OPTIONAL { ?k ifc:Unit ?unit }
    } ORDER BY ?kpi ?building""")

print(f"and read straight out of the graph, without a model — {len(recorded)} KPI(s):\n")
for row in recorded:
    print(f"  {row['kpi']:<28}{row['building']:<16}"
          f"{float(row['value']):>12,.2f} {row['unit']}")

graph.serialize(destination=str(OUTPUT / "harbourside_edited.ttl"), format="turtle")
print(f"\n{len(graph)} triples written to output/harbourside_edited.ttl")
print("output/harbourside.ttl - the twin as built - is untouched")

bot> The Science Block has the highest electricity use intensity at 44.8645. The Library has an intensity of 28.6419, and the Sports Hall has an intensity of 15.6369.
     intent=question, 0 database(s) opened

and read straight out of the graph, without a model — 6 KPI(s):

  AnnualElectricityUse        Library            27,496.22 kWh
  AnnualElectricityUse        Science Block      22,656.58 kWh
  AnnualElectricityUse        Sports Hall        14,151.35 kWh
  ElectricityUseIntensity     Library                28.64 kWh/m2
  ElectricityUseIntensity     Science Block          44.86 kWh/m2
  ElectricityUseIntensity     Sports Hall            15.64 kWh/m2

521 triples written to output/harbourside_edited.ttl
output/harbourside.ttl - the twin as built - is untouched


Two files that can be diffed, for the same reason tutorials 04 and 06 keep two: an edit is the
model's work, and overwriting the source would leave nothing to compare it against.
`Cycle.TwinChat` enforces that — `savePath` may not be the file the graph was read from.

Note also which numbers the follow-up answered from. It did not re-open a database; it read the
figures recorded a moment ago, and they are identical to
[tutorial 07's](../07-integrate-graph-and-timeseries/integrate-graph-and-timeseries.ipynb) section
9, computed there from the same rows by the same function with a hand-written plan.

## 7. The terminal chat

`Cycle.TwinChat` is the loop around all of that, and the only thing here that reads a keyboard. It
blocks on input, so it is described rather than run:

```python
from btwin import RDF, Cycle

graph = RDF.ByTTL("output/harbourside.ttl", baseIRI="https://example.org/harbourside/")

session = Cycle.TwinChat(
    graph,
    basePath=".",                                # what the graph's FilePath values resolve against
    baseIRI="https://example.org/harbourside/",  # so a derived KPI lands beside its building
    notes=NOTES,
    savePath="output/harbourside_session.ttl",   # never the file the graph was read from
)
print(session["edits"], session["saved"])
```

It shows every proposed change and asks before it lands, keeps the history and the schema threaded
from turn to turn, and auto-saves the moment a graph edit is confirmed — so a session cannot end
with a derived KPI lost. A database edit needs no saving: it was committed to the file it was
rehearsed in, because a database *is* the file.

The commands, typed at the prompt:

| | |
|---|---|
| `/sparql`, `/sql` | the locator, query or update behind the last answer |
| `/rows`, `/targets` | the readings it was written from, and the databases they came from |
| `/kpis` | what the last derive turn proposed |
| `/history`, `/schema` | the conversation as the router sees it, and the grounding block |
| `/cost`, `/silent`, `/verbose` | what it has cost; how much to show per turn |
| `/save`, `/exit` | write the graph; leave |

`myproject/chat_with_twin.py` in this repository is a worked use case around it — the same loop
with a session log that writes out every model call, every validator verdict, which files the
locator resolved to, the rows each database returned and the arithmetic behind every KPI. It
instruments the library by wrapping it rather than editing it, so the log describes the library as
it really runs.

## What this run cost

In [11]:
total = meter.Total()
print(f"{total['calls']} call(s)")
print(f"  prompt     {total['promptTokens']:>7} tokens")
print(f"  completion {total['completionTokens']:>7} tokens")
print(f"  cost       {CostMeter.Format(total['cost'])}"
      + ("  (estimated)" if total["estimated"] else ""))

print(f"\n  {'agent':<16}{'calls':>6}{'prompt':>10}{'reply':>9}{'cost':>13}")
for agent in dict.fromkeys(call["agent"] for call in meter.calls):
    window = [call for call in meter.calls if call["agent"] == agent]
    print(f"  {agent or '(none)':<16}{len(window):>6}"
          f"{sum(c['promptTokens'] for c in window):>10}"
          f"{sum(c['completionTokens'] for c in window):>9}"
          f"{CostMeter.Format(sum(c['cost'] for c in window)):>13}")

36 call(s)
  prompt       73507 tokens
  completion    3434 tokens
  cost       $0.008266

  agent            calls    prompt    reply         cost
  router half         11     31681       68    $0.003012
  agent 1 locate       4     13882     1540    $0.001729
  agent 3 write        7     20433      970    $0.002431
  answer               7      3021      479    $0.000494
  router               5      3272      191    $0.000404
  router edit          1       342        6    $0.000037
  agent 5 plan         1       876      180    $0.000160


The breakdown is worth reading rather than skipping. `router` and `router half` are pure overhead —
they retrieve nothing — and they are the price of one entry point that takes any question. The
agents that do the work are `agent 1 locate`, `agent 3 write`, `agent 5 plan` and `answer` — four,
because that is the whole model surface of a twin turn. `agent 2 repair` and `agent 4 repair` are
not a fifth and a sixth, and this run shows neither: they are those same agents handed a
validator's complaint, and a call charged to them is a query that did not compile, or did not
resolve to a file, first time. An empty repair row is a run in which nothing had to be rewritten.

## What you have

In `output/`:

| | |
|---|---|
| `databases/HB-*.db` | six SQLite files of monthly readings |
| `harbourside.ttl` | the twin as built — the estate, and the documents pointing at those files |
| `harbourside_edited.ttl` | the same graph with the KPIs the conversation derived |

And three notebooks that line up column by column:

| | Graph | Readings | Twin |
|---|---|---|---|
| describe | `RDF.SchemaSummary` | `Observation.SQLiteSchemaSummary` | both, plus `Tool.TwinPropertyBlock` |
| ask | `Cycle.RDFQueryByPrompt` | `Cycle.SQLiteQueryByPrompt` | `Cycle.TwinQueryByPrompt` |
| check | `SPARQL.Validate` | `SQL.Validate` | both, plus `Tool.TwinTargets` on the files |
| change | `Cycle.RDFEditByPrompt` | `Cycle.SQLiteEditByPrompt` | those two, plus `derive` |
| converse | `Cycle.RDFChat` | `Cycle.SQLiteChat` | `Cycle.TwinChat` |
| notebook | [04](../04-chat-with-graph/chat-with-graph.ipynb) | [06](../06-chat-with-timeseries/chat-with-timeseries.ipynb) | this one |

The right-hand column is not a third system. It is the first two, plus one node type that says
where a database is.

## Questions to try

1. **Ask for something the readings cannot support.** "What is each building's water use per
   person?" routes to `both` and runs. The occupant counts are in the graph and the readings are
   real — is the KPI meaningful? (Section 2 of tutorial 07 says why not.)
2. **Watch a locator fail.** Move one database file, then ask a question about that building.
   `Tool.TwinTargets` reports the path it could not find, `Tool.TwinRepairLocator` gets one attempt
   at it, and the question then falls back to the graph with `fellBack=True`.
3. **Record the same KPIs twice.** Re-run section 6. The confirmation now shows `was ...` beside
   each figure, because `Tool.TwinKPIObjects` was given a `baseIRI` and could see what it would
   replace.
4. **Refuse an edit.** Return `False` from `confirm` and then ask "what did you want to change?" —
   the refusal is in the transcript, so the `talk` intent can answer it.
5. **Strip the notes.** Re-run section 3 with `notes=None`. The sensor codes stop meaning anything;
   does the SQL still filter to the right rows?

## Where to go next

- [`../07-integrate-graph-and-timeseries/`](../07-integrate-graph-and-timeseries/integrate-graph-and-timeseries.ipynb)
  — every step of this pipeline, driven by hand, with no model anywhere.
- [`../04-chat-with-graph/`](../04-chat-with-graph/chat-with-graph.ipynb) and
  [`../06-chat-with-timeseries/`](../06-chat-with-timeseries/chat-with-timeseries.ipynb) — the two
  halves as conversations of their own.
- `myproject/create_twin.py` and `myproject/chat_with_twin.py` — the same two notebooks at estate
  scale: five buildings, ten databases, and a full session log.